## Broken link projections

Assumptions:
1. Some users will immediately click on the link out once they find a record of interest
2. Some users will actually further explore the data record
3. For the users that explore the data record, they will click on some of the links
4. The more records an individual user inspects, the fewer links they will actually click on

Projections based on:
1. An average number of 6 links per record
2. A broken link rate of 0.46%
3. Additional page load time of 6 seconds per record for dynamically checked pages

In [1]:
import pandas as pd

#### Variables

## Number of users
users = [1,10,100,1000,10000,100000]

## Number of records inspected per user
records_per_user = [1,10,100]

## Percentage drop off (ie- link out to record at resource)
drop_off = [1,0.75,0.5,0.25,0]

## Percentage explore (likelihood of actually clicking around the record)
explore = [1,0.75,0.5,0.25,0]

## links explored (percentage of the links actually explored)
links_clicked = [1,5/6,4/6,3/6,2/6,1/6]

## broken link rate
broken_rate = 0.0046

## average links per record
avg_links_per_record = 6

In [2]:
## calculating the penalty rates assuming and increased drop off of 25% per 10 records (60 seconds)
penalty_rate_10 = 0.25
penalty_rate = 0.25
i=2
while i<11:
    penalty_rate = penalty_rate+(penalty_rate_10)**i
    print('penalties applied: ',i,", penalty rate: ",penalty_rate)
    i=i+1

penalties applied:  2 , penalty rate:  0.3125
penalties applied:  3 , penalty rate:  0.328125
penalties applied:  4 , penalty rate:  0.33203125
penalties applied:  5 , penalty rate:  0.3330078125
penalties applied:  6 , penalty rate:  0.333251953125
penalties applied:  7 , penalty rate:  0.33331298828125
penalties applied:  8 , penalty rate:  0.3333282470703125
penalties applied:  9 , penalty rate:  0.3333320617675781
penalties applied:  10 , penalty rate:  0.33333301544189453


In [7]:
## penalty (increased drop off per additional minute)
penalty_rate_10 = 0.25
penalty_rate_100 = 0.33

projectionslist = []
for user in users:
    for records in records_per_user:
        potential_links_available = user*records*avg_links_per_record
        if records == 1:
            penalties = 0
        elif records == 10:
            penalties = penalty_rate_10
        else:
            penalties = penalty_rate_100
        for percent_explore in explore:
            total_percent_explore = percent_explore - penalties
            if total_percent_explore < 0:
                applied_percent_explore = 0
            else:
                applied_percent_explore = total_percent_explore
            records_explored = user*records*applied_percent_explore
            for clickthrough in links_clicked:
                broken_links_encountered = records_explored*avg_links_per_record*clickthrough*broken_rate
                broken_link_encounter_rate = broken_links_encountered/potential_links_available
                tmpdict = {"no. of users":user,"records inspected per user":records,
                           "total records inspected":user*records,
                           "interaction rate":percent_explore,
                           "penalty_rate_applied":penalties,
                           "exit rate":1-applied_percent_explore,
                           "records explored":records_explored,
                           "link click rate":clickthrough,
                           "potential link encounters":potential_links_available,
                           "broken links encountered": broken_links_encountered,
                           "broken link encounter rate": broken_link_encounter_rate,
                           "total additional page load time (s)":user*records*6 }
                if tmpdict["records inspected per user"] == 1:
                    projectionslist.append(tmpdict)
                elif tmpdict["records inspected per user"] == 10:
                    if tmpdict["link click rate"] <1:
                        projectionslist.append(tmpdict)
                else:
                    if tmpdict["link click rate"] < 2/3:
                        projectionslist.append(tmpdict)

projectionsdf = pd.DataFrame(projectionslist)

projectionsdf.to_csv('projections.tsv',sep='\t',header=True)